In [1]:
# Load required modules
import numpy as np
from datetime import date, datetime, timedelta
from matplotlib import pyplot as plt
import os
import xarray as xr
import pandas as pd
from tqdm import tqdm

In [3]:
df_TARA = pd.read_csv("C:\\users\\Hamid\\THESIS\\Tara_NOAA_locations.csv")

print(df_TARA.shape)
# print(df_TARA.head())

df_TARA['location'] = list(zip(df_TARA['lat'], df_TARA['lon']))

# Convert the "date" column to datetime format
df_TARA['date/time'] = pd.to_datetime(df_TARA['date/time'])

print(df_TARA.head())

(4525, 94)
   ammonium  ammonium std   nitrite  nitrite std   nitrate    age  \
0  0.017782      0.003814  0.033199     0.021603  0.006594  120.0   
1  0.019857      0.004334  0.022540     0.003445  0.004371  120.0   
2  0.019621      0.001549  0.049836     0.016420  0.008997  120.0   
3  0.019621      0.001549  0.049836     0.016420  0.008997  120.0   
4  0.019621      0.001549  0.049836     0.016420  0.008997  120.0   

   depth bathy                                        bg province  \
0      -4948.0  [NADR] North Atlantic Drift Province (MRGID:21...   
1      -3559.0  [NAST-E] North Atlantic Subtropical Gyral Prov...   
2      -3316.0  [NAST-E] North Atlantic Subtropical Gyral Prov...   
3      -3316.0  [NAST-E] North Atlantic Subtropical Gyral Prov...   
4      -3916.0  [NAST-E] North Atlantic Subtropical Gyral Prov...   

   chlorophyll a           date/time  ... sea surface temp grad  \
0        43.1353 2009-09-07 15:30:00  ...               0.39136   
1        43.6517 2009-09-

### All Marine Heatwaves detected at sample locations from 1981-2013

In [4]:
df_mhws = pd.read_csv("C:\\users\\Hamid\\THESIS\\SST\\Tara_mhws_info1.csv")

print(df_mhws.head())
print(df_mhws.shape)

print(df_mhws['n_events'].max())
print(df_mhws['location'].value_counts())
# df_mhws.sort_values(by='date_end')
# print(df_mhws.tail())

   time_start  time_end  time_peak  date_start    date_end   date_peak  \
0      724314    724318     724317  1984-02-08  1984-02-12  1984-02-11   
1      725632    725639     725638  1987-09-18  1987-09-25  1987-09-24   
2      726143    726147     726147  1989-02-10  1989-02-14  1989-02-14   
3      726234    726245     726244  1989-05-12  1989-05-23  1989-05-22   
4      726253    726258     726255  1989-05-31  1989-06-05  1989-06-02   

   index_start  index_end  index_peak  duration  ...  \
0          890        894         893         5  ...   
1         2208       2215        2214         8  ...   
2         2719       2723        2723         5  ...   
3         2810       2821        2820        12  ...   
4         2829       2834        2831         6  ...   

   intensity_cumulative_relThresh  intensity_max_abs  intensity_mean_abs  \
0                        4.447001          14.450000           14.094000   
1                        3.752099          19.980000           19.

### MHWs from 2009-2013

In [5]:
# Convert the "date" columns to datetime format
df_mhws['date_start'] = pd.to_datetime(df_mhws['date_start'])
df_mhws['date_end'] = pd.to_datetime(df_mhws['date_end'])

# Subset the DataFrame based on the condition
subset_df = df_mhws[(df_mhws['date_start'].dt.year >= 2009) & (df_mhws['date_start'].dt.year <= 2013)]

# Print the subset DataFrame
# print(subset_df)
print(subset_df.shape)

# print(subset_df['location'].value_counts())
print(subset_df['duration'].max())
print(subset_df.sort_values(by='duration'))

(7766, 31)
237
       time_start  time_end  time_peak date_start   date_end   date_peak  \
57         733572    733576     733574 2009-06-14 2009-06-18  2009-06-16   
5559       734489    734493     734489 2011-12-18 2011-12-22  2011-12-18   
23254      734466    734470     734469 2011-11-25 2011-11-29  2011-11-28   
5557       734278    734282     734281 2011-05-21 2011-05-25  2011-05-24   
23312      734008    734012     734010 2010-08-24 2010-08-28  2010-08-26   
...           ...       ...        ...        ...        ...         ...   
33767      733990    734162     734003 2010-08-06 2011-01-25  2010-08-19   
8939       733657    733838     733764 2009-09-07 2010-03-07  2009-12-23   
8711       733667    733878     733765 2009-09-17 2010-04-16  2009-12-24   
8882       733656    733870     733764 2009-09-06 2010-04-08  2009-12-23   
34097      733959    734195     733972 2010-07-06 2011-02-27  2010-07-19   

       index_start  index_end  index_peak  duration  ...  \
57          

### Match locations and dates to TARA
date_end is within 30 days before "Date" -> 9381 rows (121 locations, 92 dates)

date_peak is within 30 days before "Date" -> 8737 rows (120 locations, 89 dates)

In [6]:
# Convert date columns to datetime objects if they are not already
df_TARA['date/time'] = pd.to_datetime(df_TARA['date/time'])

subset_df = subset_df.copy()  # Create a copy of the DataFrame
subset_df['date_end'] = pd.to_datetime(subset_df['date_end'])
subset_df['date_peak'] = pd.to_datetime(subset_df['date_peak'])
subset_df['date_start'] = pd.to_datetime(subset_df['date_start'])

# Convert the 'location' column to strings in both dataframes
df_TARA['location'] = df_TARA['location'].astype(str)
subset_df['location'] = subset_df['location'].astype(str)

print(f"Shape of df_TARA: {df_TARA.shape} \nShape of subset_df: {subset_df.shape}")

# Merge dataframes based on location
merged_df = pd.merge(df_TARA, subset_df, on='location')

print()
print(f"Shape of merged_df: {merged_df.shape}")
# print(merged_df['Sample ID'].value_counts())

# print(merged_df['location'].value_counts())

# Filter rows where date_end is within 30 days before "Date"
filtered_df = merged_df[(merged_df['date_peak'] >= (merged_df['date/time'] - timedelta(days=30))) & (merged_df['date_peak'] <= merged_df['date/time'])]
filtered_df1 = merged_df[(merged_df['date_end'] >= (merged_df['date/time'] - timedelta(days=30))) & (merged_df['date_end'] <= merged_df['date/time'])]


filtered_df = filtered_df.reset_index(drop=True)
filtered_df1 = filtered_df1.reset_index(drop=True)

# print(filtered_df)


filtered_df1.to_csv("C:\\users\\Hamid\\THESIS\\mhws_TARA_filtered_1.csv", index=False)

print(filtered_df.columns)
print(f"Shape of filtered_df: {filtered_df.shape}")
print(f"Shape of filtered_df1: {filtered_df1.shape}")

print(filtered_df['date/time'].iloc[0])
print(filtered_df['date_start'].iloc[0])
print(filtered_df['date_peak'].iloc[0])
print(filtered_df['date_end'].iloc[0])


filtered_df['time_start_days'] = (filtered_df['date/time'] - filtered_df['date_start']).dt.days
filtered_df['time_peak_days'] = (filtered_df['date/time'] - filtered_df['date_peak']).dt.days
filtered_df['time_end_days'] = (filtered_df['date/time'] - filtered_df['date_end']).dt.days


df999 = filtered_df[['date/time', 
                     #'date_start', 'date_peak', 'date_end',
                     'time_start_days', 'time_peak_days',
                                'time_end_days',
                                ]]

print(df999.describe())


filtered_df1['time_start_days'] = (filtered_df1['date/time'] - filtered_df1['date_start']).dt.days
filtered_df1['time_peak_days'] = (filtered_df1['date/time'] - filtered_df1['date_peak']).dt.days
filtered_df1['time_end_days'] = (filtered_df1['date/time'] - filtered_df1['date_end']).dt.days


df666 = filtered_df1[['date/time', 'time_start_days', 'time_peak_days',
                                'time_end_days',
                                ]]

print(df666.describe())



#print(filtered_df['Sample ID'].value_counts())
print(filtered_df['category'].value_counts())
print(filtered_df['location'].value_counts())
print(subset_df['duration'].max())
print(filtered_df['duration'].max())
# print("------------")
#print(filtered_df1['Sample ID'].value_counts())
print(filtered_df1['category'].value_counts())

print(filtered_df1['location'].value_counts())
print(filtered_df1['duration'].max())

Shape of df_TARA: (4525, 94) 
Shape of subset_df: (7766, 31)

Shape of merged_df: (62636, 124)
Index(['ammonium', 'ammonium std', 'nitrite', 'nitrite std', 'nitrate', 'age',
       'depth bathy', 'bg province', 'chlorophyll a', 'date/time',
       ...
       'intensity_var_relThresh', 'intensity_cumulative_relThresh',
       'intensity_max_abs', 'intensity_mean_abs', 'intensity_var_abs',
       'intensity_cumulative_abs', 'category', 'rate_onset', 'rate_decline',
       'n_events'],
      dtype='object', length=124)
Shape of filtered_df: (955, 124)
Shape of filtered_df1: (1129, 124)
2009-09-15 11:30:00
2009-08-16 00:00:00
2009-08-20 00:00:00
2009-09-01 00:00:00
                           date/time  time_start_days  time_peak_days  \
count                            955       955.000000      955.000000   
mean   2011-02-27 06:18:47.937172480        19.934031       14.860733   
min              2009-09-15 11:30:00         2.000000        0.000000   
25%              2010-04-04 03:04:00  

## Add "MHWs" and "category" column to Tara data (for dashboard)

In [7]:
#filtered_df = pd.read_csv("mhws_TARA_filtered1.csv")

df = pd.read_csv("cleanest - 4066 obs.csv")

# print(filtered_df[filtered_df['category']=='Strong'])


# Get the counts of Sample IDs in filtered_df
sample_counts = filtered_df1['sample_id'].value_counts()


# Group filtered_df by Sample ID and aggregate categories into lists
sample_info = filtered_df1.groupby('sample_id')['category'].agg(list).reset_index()

# Create a new dataframe with all unique Sample IDs from df
count_df = pd.DataFrame({'sample_id': df['sample_id'].unique()})

# Count occurrences of each Sample ID in small_df
count_df['mhw_count'] = count_df['sample_id'].map(sample_counts).fillna(0).astype(int)

# Merge category information to count_df
count_df = pd.merge(count_df, sample_info, on='sample_id', how='left')

# print(count_df)
# print(count_df['MHWs'].value_counts())


df = pd.merge(df, count_df, on='sample_id')

df['mhw_category'] = df['category'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)

print(df['mhw_category'].value_counts())

# Replace values in 'category'
category_replace = {
    'Moderate, Moderate': 'Moderate',
    'Moderate, Strong': 'Strong',
    'Strong, Moderate': 'Strong',
    'Moderate, Moderate, Moderate': 'Moderate',
}


# Apply replacements to 'category'
df['mhw_category'] = df['mhw_category'].replace(category_replace)

df.fillna({'mhw_category' : "none"}, inplace=True)

print(df.shape)

df = df.drop(columns="category", inplace=False)

print(df.shape)

print(df['mhw_category'].value_counts())


print(df['mhw_count'].value_counts())
# print(df.shape)
# # print(df)

df.to_csv("TARA_mhws_3.csv", index=False)

mhw_category
Moderate                        497
Strong                          135
Moderate, Moderate              101
Severe                           31
Moderate, Strong                 27
Strong, Moderate                 23
Moderate, Moderate, Moderate      7
Name: count, dtype: int64
(4066, 95)
(4066, 94)
mhw_category
none        3245
Moderate     605
Strong       185
Severe        31
Name: count, dtype: int64
mhw_count
0    3245
1     663
2     151
3       7
Name: count, dtype: int64
